# Bagging classifier
BaggingClassifier on a synthetic dataset of 10,000 rows: bagging, bagged SVMs, pasting, random subspaces,
random patches, the out-of-bag score and a grid search. The decision-surface demo is `app.py`
(`python app.py`, then open http://127.0.0.1:8050).

In [ ]:
import time

import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 1. Data and a single tree

In [ ]:
# 10,000 rows, 10 columns, of which 3 carry information
X, y = make_classification(n_samples=10000, n_features=10, n_informative=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("single decision tree:", accuracy_score(y_test, dt.predict(X_test)))

## 2. Bagging: 500 trees, each on 25% of the rows, drawn with replacement
In scikit-learn 1.9 the base model is passed as `estimator` (older code used `base_estimator`).

In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,      # 25% of 8,000 rows = 2,000 rows per tree
    bootstrap=True,        # with replacement
    random_state=42,
    n_jobs=-1,             # use every CPU core
)
bag.fit(X_train, y_train)
print("bagging:", accuracy_score(y_test, bag.predict(X_test)))

In [ ]:
# Which rows and columns did the first tree get?
print("rows for tree 1:", bag.estimators_samples_[0].shape, "first few:", bag.estimators_samples_[0][:8])
print("columns for tree 1:", bag.estimators_features_[0])

In [ ]:
# The same with max_samples=0.5: 4,000 rows per tree
bag_half = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.5, random_state=42, n_jobs=-1)
bag_half.fit(X_train, y_train)
print("bagging, 50% rows:", accuracy_score(y_test, bag_half.predict(X_test)))
print("rows for tree 1:", bag_half.estimators_samples_[0].shape)

## 3. Bagging SVMs

In [ ]:
bag_svm = BaggingClassifier(estimator=SVC(), n_estimators=500, max_samples=0.25, bootstrap=True,
                            random_state=42, n_jobs=-1)
bag_svm.fit(X_train, y_train)
print("bagging with SVM:", accuracy_score(y_test, bag_svm.predict(X_test)))

## 4. Pasting: rows without replacement (bootstrap=False); verbose=1 prints progress

In [ ]:
paste = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.25, bootstrap=False,
                          random_state=42, verbose=1, n_jobs=-1)
paste.fit(X_train, y_train)
print("pasting:", accuracy_score(y_test, paste.predict(X_test)))

## 5. Random subspaces: all rows, half the columns

In [ ]:
subspace = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=1.0, bootstrap=False,
                             max_features=0.5, bootstrap_features=True, random_state=42, n_jobs=-1)
subspace.fit(X_train, y_train)
print("random subspaces:", accuracy_score(y_test, subspace.predict(X_test)))
print("rows for tree 1:", subspace.estimators_samples_[0].shape, "| columns for tree 1:", subspace.estimators_features_[0])

## 6. Random patches: 25% of the rows and half the columns

In [ ]:
patches = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.25, bootstrap=True,
                            max_features=0.5, bootstrap_features=True, random_state=42, n_jobs=-1)
patches.fit(X_train, y_train)
print("random patches:", accuracy_score(y_test, patches.predict(X_test)))

## 7. Out-of-bag score
Each tree is scored on the training rows its sample missed; oob_score_ averages these into one accuracy.

In [ ]:
bag_oob = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=0.25, bootstrap=True,
                            oob_score=True, random_state=42, n_jobs=-1)
bag_oob.fit(X_train, y_train)
print("out-of-bag score:", round(bag_oob.oob_score_, 4))
print("test accuracy:   ", accuracy_score(y_test, bag_oob.predict(X_test)))

## 8. Grid search over the bagging hyperparameters

In [ ]:
parameters = {
    "n_estimators": [50, 100, 500],
    "max_samples": [0.1, 0.4, 0.7, 1.0],
    "bootstrap": [True, False],
    "max_features": [0.1, 0.4, 0.7, 1.0],
}
start = time.time()
search = GridSearchCV(BaggingClassifier(random_state=42), parameters, cv=5, n_jobs=-1)
search.fit(X_train, y_train)
print(f"{len(search.cv_results_['params'])} combinations in {time.time() - start:.0f} s")
print("best parameters:", search.best_params_)
print("best cross-validation accuracy:", round(search.best_score_, 4))
print("test accuracy:", search.score(X_test, y_test))

## 9. The decision-surface demo (same code as app.py)

In [ ]:
from app import figure, fit

(single, a1), (bagged, a2) = fit("decision tree", n_estimators=100, max_samples=50)
figure([(f"single tree: {a1:.3f}", single), (f"bagging: {a2:.3f}", bagged)])